# Weighted S10000 K-Tilde / Lambda Comparison

This notebook audits the four S10000 source laws used by the weighted experiment suite. It converts stored Fourier energies to the unitary convention and computes Lambda tables with

$$\mu_{1/2}(i)=\tfrac12\widetilde\mu_{\mathrm{S10000}}(i)+\tfrac{1}{2n}.$$

The five-trial convergence study keeps raw K-tilde errors while applying $\zeta=1/2$ to its sampling-law and Lambda diagnostics. Figures are written under `results/weighted/figures/`.


In [ ]:
from pathlib import Path
import sys

import importlib
import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_DIR = Path.cwd()
for candidate in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    helper_candidates = [
        candidate / 'sd15_conditioning_experiment.py',
        candidate / 'sd1.5' / 'analyze_results' / 'sd15_conditioning_experiment.py',
    ]
    for helper_path in helper_candidates:
        if helper_path.is_file():
            helper_dir = helper_path.parent
            if str(helper_dir) not in sys.path:
                sys.path.insert(0, str(helper_dir))
            break
    else:
        continue
    break
else:
    raise FileNotFoundError('Could not find sd15_conditioning_experiment.py from the notebook cwd.')

import sd15_conditioning_experiment as exp
exp = importlib.reload(exp)

SD15_ROOT = exp.find_sd15_root(NOTEBOOK_DIR)
S10000_CONFIG = SD15_ROOT / 'ktilde' / 'weighted' / 'config_convergence.json'
ZETA = 0.5
CATALOG = exp.load_ktilde_catalog(SD15_ROOT, config_path=S10000_CONFIG)
TABLES = exp.build_lambda_tables(
    SD15_ROOT,
    config_path=S10000_CONFIG,
    probability_regularization_zeta=ZETA,
    skip_missing=True,
)
BANK_SUMMARY = pd.DataFrame(
    [
        {
            'name': name,
            'role': value.get('role', name),
            'label': value.get('label', name),
            'christoffel_law': value.get('christoffel_law', ''),
            'prompt': value.get('prompt', ''),
            'prompt_bank': ', '.join(value.get('prompt_bank', [])),
            'artifact_exists': (SD15_ROOT / 'ktilde' / 'weighted' / 'reference' / f'{name}.npz').is_file(),
        }
        for name, value in CATALOG.items()
    ]
)


def style_plain_numbers(frame):
    return frame.style.format(lambda value: exp.format_plain_number(value))


display(BANK_SUMMARY)
if TABLES['missing_names']:
    print('Missing k-tilde artifacts:', TABLES['missing_names'])
FFT_ENERGY_SCALES = sorted({int(round(value)) for value in TABLES['fft_energy_scale'].values()})
display(
    Markdown(
        "**Weighted source-law audit.** "
        "Stored `K_tilde` artifacts were estimated with the unnormalized FFT, so the absolute lambda and kappa tables below divide Fourier energies by `H * W` "
        f"({', '.join(str(value) for value in FFT_ENERGY_SCALES)}) to report the unitary-FFT convention. "
        "Every sampling column uses zeta=1/2 regularization; this changes the absolute compatibility values."
    )
)
display(TABLES['kappa_df'].style.format({'kappa_hat': exp.format_plain_number}))
display(
    TABLES['lambda_df'].style.format(
        {
            'lambda_hat': exp.format_plain_number,
            'kappa_hat': exp.format_plain_number,
            'mismatch_penalty': exp.format_plain_number,
        }
    )
)
display(style_plain_numbers(TABLES['lambda_table']))
display(style_plain_numbers(TABLES['penalty_table']))
display(
    TABLES['matched_check'].style.format(
        {
            'lambda_hat': exp.format_plain_number,
            'kappa_hat': exp.format_plain_number,
            'abs_lambda_minus_kappa': exp.format_plain_number,
        }
    )
)
display(
    Markdown(
        r"**$\widetilde{\Lambda}'$ interpretation.** "
        r"The $\widetilde{\lambda}$ heatmap shows the absolute compatibility cost of using sampling law "
        r"$\widetilde{\mu}_{c_s}$ with Christoffel function $c_r$. "
        r"$\widetilde{\Lambda}'$ is the row-normalized mismatch factor "
        r"$\widetilde{\lambda}(c_r,c_r,c_s)/\widetilde{\kappa}(c_r)$, so values near 1 mean the sampling law is close to the matched baseline for that row, while larger values show how much extra penalty you pay from mismatch."
    )
)


## Five-Trial Algorithm 1 K-Tilde Convergence

The four saved S10000 artifacts remain fixed references. For every prompt, five
new S10000 estimates use disjoint latent-seed blocks and record relative
$\ell_2$ error, relative $\ell_\infty$ error, the regularized Lambda
max-ratio, and the regularized maximum log-probability ratio every 10
iterations. The probability-based metrics use $\zeta=1/2$.

Each curve is the arithmetic mean of the five trial values at that iteration.
Shading is the 95% Student-$t$ confidence interval computed on the original
metric scale; the mean and bounds are then displayed on the existing
logarithmic $y$-axis. The completion table remains visible while any of the 20
jobs is missing or incomplete.


In [ ]:
TRIAL_COMPLETION = exp.ktilde_convergence_trial_completion_table(SD15_ROOT)
CONVERGENCE_FIGURE_DIR = SD15_ROOT / 'results' / 'weighted' / 'figures' / 'ktilde_convergence'
CONVERGENCE_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
COMPLETION_PATH = CONVERGENCE_FIGURE_DIR / 'ktilde_convergence_five_trial_completion.csv'
TRIAL_COMPLETION.to_csv(COMPLETION_PATH, index=False)
display(TRIAL_COMPLETION)

CONVERGENCE_FIGURE_PATHS = {}
if not TRIAL_COMPLETION['status'].eq('complete').all():
    print('The five-trial convergence figures are pending. Run scripts/weighted/ktilde_convergence/list_all.sh to inspect all 20 jobs.')
else:
    CONVERGENCE_TRACES = exp.load_ktilde_convergence_trial_traces(SD15_ROOT)
    CONVERGENCE_SUMMARIES = exp.summarize_ktilde_convergence_trials(
        CONVERGENCE_TRACES,
        confidence_level=0.95,
    )
    CONVERGENCE_FIGURE_PATHS = exp.export_ktilde_convergence_trial_figure_set(
        CONVERGENCE_SUMMARIES,
        output_dir=CONVERGENCE_FIGURE_DIR,
        file_format='pdf',
        metrics=list(exp.KTILDE_TRIAL_CONVERGENCE_METRICS),
        show=True,
    )
    display(pd.Series({key: str(value) for key, value in CONVERGENCE_FIGURE_PATHS.items()}))
CONVERGENCE_FIGURE_PATHS


## Export Lambda Figures

This cell exports the absolute Lambda heatmap, the individual sampling-law
plots, and the compact sampling-law row. The displayed `FIGURE_PATHS` series
is the checklist of generated paper figures.


In [ ]:
FIGURE_DIR = SD15_ROOT / 'results' / 'weighted' / 'figures' / 'lambda_figures'
FIGURE_PATHS = exp.export_lambda_figure_set(
    TABLES,
    output_dir=FIGURE_DIR,
    file_format='pdf',
    show=True,
)
display(pd.Series({key: str(value) for key, value in FIGURE_PATHS.items()}))
FIGURE_PATHS


## Regularized S10000 Probability Audit

This audit records both the raw artifact statistics and the exact $\zeta=1/2$ law used by reconstruction and Lambda analysis.


In [ ]:
import numpy as np

probability_rows = []
for role, info in TABLES['bank'].items():
    raw = np.asarray(info['raw_probabilities'], dtype=np.float64).reshape(-1)
    effective = np.asarray(info['probabilities'], dtype=np.float64).reshape(-1)
    expected = 0.5 * raw + 0.5 / raw.size
    np.testing.assert_allclose(effective, expected, rtol=0.0, atol=2e-16)
    probability_rows.append(
        {
            'role': role,
            'artifact': info['name'],
            'n': raw.size,
            'zeta': info['probability_regularization_zeta'],
            'raw_sum': raw.sum(),
            'raw_min': raw.min(),
            'raw_max': raw.max(),
            'regularized_sum': effective.sum(),
            'regularized_min': effective.min(),
            'regularized_max': effective.max(),
            'required_floor': 0.5 / raw.size,
            'floor_satisfied': bool(effective.min() >= 0.5 / raw.size),
        }
    )
PROBABILITY_AUDIT = pd.DataFrame(probability_rows)
AUDIT_PATH = SD15_ROOT / 'results' / 'weighted' / 'figures' / 'ktilde_lambda' / 'probability_audit.csv'
AUDIT_PATH.parent.mkdir(parents=True, exist_ok=True)
PROBABILITY_AUDIT.to_csv(AUDIT_PATH, index=False)
display(PROBABILITY_AUDIT)
AUDIT_PATH
